# Deep CFR Complex-Spec Exact-Evaluation Preflight

Before running the wider-network architecture screen on `r4_s4_h2_hp2pt_ss`, this notebook answers two operational questions:

1. How long do neural-to-dense compilation and exact best response take separately?
2. Is one exact evaluation safe within the machine's available RAM?

The exact evaluation runs in an isolated CPU subprocess. If it is killed by the OS or exceeds the configured timeout, the notebook kernel and trained policy snapshot remain intact.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import torch

from liars_poker.algo.deep_cfr import DeepCFRTrainer
from liars_poker.core import GameSpec
from liars_poker.policies.neural import NeuralMLP
from liars_poker.serialization import save_policy

assert torch.cuda.is_available(), 'The snapshot-generation step requires CUDA.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CPU count:', os.cpu_count())
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

ADVANTAGE_HIDDEN_SIZES = (1024, 1024)
STRATEGY_HIDDEN_SIZES = (512, 512)
TRAINING_SECONDS = 60
TRAVERSALS_PER_PLAYER = 1024
TRAVERSAL_BATCH_SIZE = 1024
FITTING_BATCH_SIZE = 4096
SEED = 7

RUN_ROOT = REPO_ROOT / 'artifacts' / 'deep_cfr_eval_preflight' / spec.to_short_str()
SNAPSHOT_DIR = RUN_ROOT / 'learned_average_snapshot'
STATUS_PATH = RUN_ROOT / 'evaluation_status.json'
RESULT_PATH = RUN_ROOT / 'evaluation_result.json'
EVALUATION_TIMEOUT_MINUTES = 240

RUN_ROOT.mkdir(parents=True, exist_ok=True)
spec.to_short_str(), RUN_ROOT

## Generate one representative learned-average snapshot

Only the advantage networks are widened to `1024x1024`; the deployable average-strategy networks remain `512x512`. No dense policy or exact average is constructed during training.

In [ ]:
trainer = DeepCFRTrainer(
    spec,
    hidden_sizes=STRATEGY_HIDDEN_SIZES,
    device=device,
    seed=SEED,
    advantage_buffer_capacity=1_000_000,
    strategy_buffer_capacity=1_000_000,
    learning_rate=1e-3,
    batch_size=FITTING_BATCH_SIZE,
    advantage_train_steps=100,
    strategy_train_steps=50,
    advantage_positive_weight=0.5,
    strategy_weighting='linear',
    highest_regret_fallback=True,
    alternating_updates=True,
    validation_fraction=0.0,
    traversal_backend='gpu_native',
    traversal_batch_size=TRAVERSAL_BATCH_SIZE,
    device_replay=True,
    fused_optimizer=True,
    amp_dtype=None,
    compile_models=False,
)

torch.manual_seed(SEED + 10_000)
trainer.advantage_nets = [
    NeuralMLP(trainer.encoder.input_dim, trainer.encoder.action_dim, ADVANTAGE_HIDDEN_SIZES).to(device)
    for _ in range(2)
]
trainer.advantage_optimizers = [
    trainer._make_optimizer(model) for model in trainer.advantage_nets
]

training_records = []
measured_training_s = 0.0
while measured_training_s < TRAINING_SECONDS:
    start = time.perf_counter()
    record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
    measured_training_s += time.perf_counter() - start
    training_records.append(record)

snapshot_start = time.perf_counter()
save_policy(trainer.average_policy(), str(SNAPSHOT_DIR))
snapshot_s = time.perf_counter() - snapshot_start
snapshot_bytes = sum(path.stat().st_size for path in SNAPSHOT_DIR.rglob('*') if path.is_file())

training_summary = pd.DataFrame([{
    'iterations': trainer.iteration,
    'measured training s': measured_training_s,
    'mean traversal s': sum(row['timing']['traversal_s'] for row in training_records) / len(training_records),
    'mean advantage fit s': sum(row['timing']['advantage_training_s'] for row in training_records) / len(training_records),
    'mean strategy fit s': sum(row['timing']['strategy_training_s'] for row in training_records) / len(training_records),
    'snapshot save s': snapshot_s,
    'snapshot MiB': snapshot_bytes / 2**20,
}])
display(training_summary)
print('Snapshot:', SNAPSHOT_DIR)

## Isolated CPU exact-evaluation benchmark

The child process records phase transitions in `evaluation_status.json`. The parent samples the child's resident memory through Linux `/proc`, so peak memory is still reported if the evaluator fails before writing its final result.

In [ ]:
EVALUATOR_CODE = r'''
import json
import os
import resource
import sys
import time
import traceback
from pathlib import Path

repo_root = Path(sys.argv[1])
policy_dir = Path(sys.argv[2])
status_path = Path(sys.argv[3])
result_path = Path(sys.argv[4])
sys.path.insert(0, str(repo_root))

from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.policies.neural import compile_neural_to_dense
from liars_poker.serialization import load_policy

def atomic_json(path, data):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(data, indent=2), encoding='utf-8')
    tmp.replace(path)

started = time.perf_counter()
timings = {}
try:
    atomic_json(status_path, {'phase': 'loading policy', 'pid': os.getpid()})
    phase_start = time.perf_counter()
    policy, spec = load_policy(str(policy_dir))
    timings['load_policy_s'] = time.perf_counter() - phase_start

    atomic_json(status_path, {'phase': 'compiling neural policy to dense', 'pid': os.getpid(), **timings})
    phase_start = time.perf_counter()
    dense = compile_neural_to_dense(policy)
    timings['dense_compile_s'] = time.perf_counter() - phase_start

    atomic_json(status_path, {'phase': 'running exact best response', 'pid': os.getpid(), **timings})
    phase_start = time.perf_counter()
    _, meta = best_response_dense(spec, dense, debug=False, store_state_values=False)
    timings['exact_br_s'] = time.perf_counter() - phase_start
    p_first, p_second = meta['computer'].exploitability()

    result = {
        'status': 'complete',
        **timings,
        'total_evaluator_s': time.perf_counter() - started,
        'p_first': float(p_first),
        'p_second': float(p_second),
        'exploitability': float(p_first + p_second - 1.0),
        'child_reported_peak_rss_GiB': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20,
    }
    atomic_json(result_path, result)
    atomic_json(status_path, {'phase': 'complete', 'pid': os.getpid(), **result})
except BaseException as exc:
    failure = {
        'status': 'failed',
        'error': repr(exc),
        'traceback': traceback.format_exc(),
        'total_evaluator_s': time.perf_counter() - started,
        **timings,
    }
    atomic_json(result_path, failure)
    atomic_json(status_path, {'phase': 'failed', 'pid': os.getpid(), **failure})
    raise
'''


def linux_process_memory(pid):
    status_path = Path(f'/proc/{pid}/status')
    if not status_path.exists():
        return None
    values = {}
    for line in status_path.read_text(encoding='utf-8').splitlines():
        if line.startswith(('VmRSS:', 'VmHWM:')):
            key, value, _ = line.split()
            values[key.rstrip(':')] = int(value) * 1024
    return values


def linux_available_memory():
    path = Path('/proc/meminfo')
    if not path.exists():
        return None
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.startswith('MemAvailable:'):
            return int(line.split()[1]) * 1024
    return None

In [ ]:
for path in (STATUS_PATH, RESULT_PATH):
    if path.exists():
        path.unlink()

command = [
    sys.executable,
    '-u',
    '-c',
    EVALUATOR_CODE,
    str(REPO_ROOT),
    str(SNAPSHOT_DIR),
    str(STATUS_PATH),
    str(RESULT_PATH),
]
process = subprocess.Popen(
    command,
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

monitor_start = time.perf_counter()
deadline = monitor_start + EVALUATION_TIMEOUT_MINUTES * 60
peak_child_rss = 0
minimum_available_memory = None
last_phase = None
last_progress_print = monitor_start
timed_out = False

while process.poll() is None:
    now = time.perf_counter()
    memory = linux_process_memory(process.pid)
    if memory is not None:
        peak_child_rss = max(peak_child_rss, memory.get('VmRSS', 0), memory.get('VmHWM', 0))
    available = linux_available_memory()
    if available is not None:
        minimum_available_memory = available if minimum_available_memory is None else min(minimum_available_memory, available)

    status = json.loads(STATUS_PATH.read_text(encoding='utf-8')) if STATUS_PATH.exists() else {}
    phase = status.get('phase', 'starting')
    if phase != last_phase or now - last_progress_print >= 30:
        rss_gib = peak_child_rss / 2**30
        available_gib = available / 2**30 if available is not None else float('nan')
        print(f'{(now - monitor_start) / 60:7.1f} min | {phase} | peak child RSS={rss_gib:.2f} GiB | available RAM={available_gib:.2f} GiB')
        last_phase = phase
        last_progress_print = now

    if now >= deadline:
        timed_out = True
        process.kill()
        break
    time.sleep(1.0)

stdout, stderr = process.communicate()
monitor_wall_s = time.perf_counter() - monitor_start
result = json.loads(RESULT_PATH.read_text(encoding='utf-8')) if RESULT_PATH.exists() else {
    'status': 'timeout' if timed_out else 'process exited without a result',
}
result.update({
    'process return code': process.returncode,
    'parent measured wall s': monitor_wall_s,
    'parent measured peak child RSS GiB': peak_child_rss / 2**30,
    'minimum system available RAM GiB': (
        minimum_available_memory / 2**30 if minimum_available_memory is not None else None
    ),
    'timed out': timed_out,
})

display(pd.DataFrame([result]).T.rename(columns={0: 'value'}))
if stdout.strip():
    print('\nChild stdout:\n', stdout)
if stderr.strip():
    print('\nChild stderr:\n', stderr)
print('\nDurable result:', RESULT_PATH)

## Decision rule

- **Proceed with one background evaluator** if the run completes with comfortable RAM headroom and evaluation duration is operationally acceptable.
- **Allow snapshots to queue** if evaluation takes longer than the intended snapshot interval; GPU training should not wait.
- **Do not launch concurrent exact-BR workers** if one evaluation consumes a substantial fraction of the machine's 48 GiB RAM.
- **Pause the larger architecture screen** if dense compilation or exact BR approaches the memory limit. That indicates the evaluator itself must be redesigned before scaling the training experiment.